In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("../data/global_air_quality_2014_2025.csv")
df = df.drop(columns=["State", "AQI_Bucket"])
df.head()

,Country,City,Date,PM2.5 (ug/m3),PM10 (ug/m3),NO (ug/m3),NO2 (ug/m3),NOx (ppb),NH3 (ug/m3),CO (mg/m3),...,Benzene (ug/m3),Toluene (ug/m3),Xylene (ug/m3),AQI,Wind_Speed (km/h),Humidity (%),Deforestation_Rate_%,Industry_Growth_%,CO2_Emission_MT,Population_Density_per_SqKm
0,Australia,Canberra,2014-01-01,6.85,21.13,3.37,7.16,10.53,6.12,0.28,...,1.07,2.41,2.54,39.0,12.88,47.47,0.0051,0.77,7.80,343.71
1,Australia,Sydney,2014-01-01,14.88,8.24,2.46,7.41,9.87,2.63,0.24,...,0.62,2.35,0.78,84.0,25.59,71.72,0.0055,0.69,7.42,3734.49
2,Australia,Newcastle,2014-01-01,5.57,18.62,2.57,5.44,8.01,4.07,0.30,...,0.80,1.97,1.38,27.0,26.06,57.64,0.0042,1.17,12.21,371.91
3,Australia,Wollongong,2014-01-01,4.98,19.61,2.52,7.41,9.93,4.67,0.21,...,1.39,5.07,1.44,35.0,9.97,50.29,0.0025,1.16,4.09,5.00
4,Australia,Central Coast,2014-01-01,4.94,9.70,4.26,12.23,16.49,0.46,0.26,...,1.12,3.01,2.51,41.0,8.34,45.25,0.0041,1.33,5.44,5.00


In [4]:
df.shape

(331920, 22)

In [5]:
city = pd.read_csv("../data/worldcities.csv")
city["country"] = city["country"].replace("Korea, North", "North Korea")
city["country"] = city["country"].replace("Korea, South", "South Korea")

In [6]:
supplement = pd.read_csv("../data/worldcities_supplement.csv")
supplement.head()

,city,city_ascii,country,lat,lon
0,Jakarta Selatan,Jakarta Selatan,Indonesia,-6.2660,106.8135
1,Jakarta Timur,Jakarta Timur,Indonesia,-6.2521,106.8840
2,Jakarta Pusat,Jakarta Pusat,Indonesia,-6.1777,106.8403
3,Jakarta Barat,Jakarta Barat,Indonesia,-6.1676,106.7673
4,Jakarta Utara,Jakarta Utara,Indonesia,-6.1339,106.8823


In [7]:
city = city.rename(columns={"lng": "lon"})
city = city[["city", "city_ascii", "country", "lat", "lon"]]
city = pd.concat([city, supplement], ignore_index=True)
city["id"] = city["city_ascii"] + " | " + city["country"]

city = city[["id", "lat", "lon"]].drop_duplicates(subset="id")
city = city.rename(columns={"lat": "Lat", "lon": "Lon"}).set_index("id")
city.head()

,Lat,Lon
id,,
Tokyo | Japan,35.6850,139.7514
Jakarta | Indonesia,-6.1753,106.8269
Delhi | India,28.6600,77.2300
Chongqing | China,29.5500,106.5069
Guangzhou | China,23.1300,113.2600


In [8]:
city.shape

(47638, 2)

In [9]:
df["City_id"] = df["City"] + " | " + df["Country"]
df = df.drop(columns=["City", "Country"])
df.head()

,Date,PM2.5 (ug/m3),PM10 (ug/m3),NO (ug/m3),NO2 (ug/m3),NOx (ppb),NH3 (ug/m3),CO (mg/m3),SO2 (ug/m3),O3 (ug/m3),...,Toluene (ug/m3),Xylene (ug/m3),AQI,Wind_Speed (km/h),Humidity (%),Deforestation_Rate_%,Industry_Growth_%,CO2_Emission_MT,Population_Density_per_SqKm,City_id
0,2014-01-01,6.85,21.13,3.37,7.16,10.53,6.12,0.28,2.34,38.00,...,2.41,2.54,39.0,12.88,47.47,0.0051,0.77,7.80,343.71,Canberra | Australia
1,2014-01-01,14.88,8.24,2.46,7.41,9.87,2.63,0.24,2.45,60.70,...,2.35,0.78,84.0,25.59,71.72,0.0055,0.69,7.42,3734.49,Sydney | Australia
2,2014-01-01,5.57,18.62,2.57,5.44,8.01,4.07,0.30,6.63,60.82,...,1.97,1.38,27.0,26.06,57.64,0.0042,1.17,12.21,371.91,Newcastle | Australia
3,2014-01-01,4.98,19.61,2.52,7.41,9.93,4.67,0.21,2.05,45.29,...,5.07,1.44,35.0,9.97,50.29,0.0025,1.16,4.09,5.00,Wollongong | Australia
4,2014-01-01,4.94,9.70,4.26,12.23,16.49,0.46,0.26,2.91,46.39,...,3.01,2.51,41.0,8.34,45.25,0.0041,1.33,5.44,5.00,Central Coast | Australia


In [10]:
df = df.join(city, how="left", on="City_id", validate="m:1")

df["Date"] = pd.to_datetime(df["Date"])

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month

df = df.drop(columns="Date")
df = df.dropna()
df.head()

,PM2.5 (ug/m3),PM10 (ug/m3),NO (ug/m3),NO2 (ug/m3),NOx (ppb),NH3 (ug/m3),CO (mg/m3),SO2 (ug/m3),O3 (ug/m3),Benzene (ug/m3),...,Humidity (%),Deforestation_Rate_%,Industry_Growth_%,CO2_Emission_MT,Population_Density_per_SqKm,City_id,Lat,Lon,Year,Month
0,6.85,21.13,3.37,7.16,10.53,6.12,0.28,2.34,38.00,1.07,...,47.47,0.0051,0.77,7.80,343.71,Canberra | Australia,-35.2931,149.1269,2014,1
1,14.88,8.24,2.46,7.41,9.87,2.63,0.24,2.45,60.70,0.62,...,71.72,0.0055,0.69,7.42,3734.49,Sydney | Australia,-33.8678,151.2100,2014,1
2,5.57,18.62,2.57,5.44,8.01,4.07,0.30,6.63,60.82,0.80,...,57.64,0.0042,1.17,12.21,371.91,Newcastle | Australia,-32.9167,151.7500,2014,1
3,4.98,19.61,2.52,7.41,9.93,4.67,0.21,2.05,45.29,1.39,...,50.29,0.0025,1.16,4.09,5.00,Wollongong | Australia,-34.4331,150.8831,2014,1
4,4.94,9.70,4.26,12.23,16.49,0.46,0.26,2.91,46.39,1.12,...,45.25,0.0041,1.33,5.44,5.00,Central Coast | Australia,-33.2992,151.1922,2014,1


In [11]:
df.shape

(253584, 24)

In [12]:
df[["City_id"]].groupby("City_id").value_counts().sort_values()

City_id
Orenburg | Russia                144
Overland Park | United States    144
Oudtshoorn | South Africa        144
Otsu | Japan                     144
Ota | Japan                      144
                                ... 
Columbus | United States         288
Columbia | United States         288
Udaipur | India                  288
Erfurt | Germany                 288
Springfield | United States      432
Name: count, Length: 1746, dtype: int64

In [30]:
df = df.drop_duplicates(subset=["City_id", "Year", "Month"])
df[["City_id"]].groupby("City_id").value_counts().sort_values()

City_id
Aba | Nigeria            144
Abakaliki | Nigeria      144
Abakan | Russia          144
Abbottabad | Pakistan    144
Abeokuta | Nigeria       144
                        ... 
Zifta | Egypt            144
Zigong | China           144
Zonguldak | Turkey       144
Zunyi | China            144
Zwickau | Germany        144
Name: count, Length: 1745, dtype: int64

In [31]:
df.shape

(251280, 24)

In [32]:
df.head()

,PM2.5 (ug/m3),PM10 (ug/m3),NO (ug/m3),NO2 (ug/m3),NOx (ppb),NH3 (ug/m3),CO (mg/m3),SO2 (ug/m3),O3 (ug/m3),Benzene (ug/m3),...,Humidity (%),Deforestation_Rate_%,Industry_Growth_%,CO2_Emission_MT,Population_Density_per_SqKm,City_id,Lat,Lon,Year,Month
0,6.85,21.13,3.37,7.16,10.53,6.12,0.28,2.34,38.00,1.07,...,47.47,0.0051,0.77,7.80,343.71,Canberra | Australia,-35.2931,149.1269,2014,1
1,14.88,8.24,2.46,7.41,9.87,2.63,0.24,2.45,60.70,0.62,...,71.72,0.0055,0.69,7.42,3734.49,Sydney | Australia,-33.8678,151.2100,2014,1
2,5.57,18.62,2.57,5.44,8.01,4.07,0.30,6.63,60.82,0.80,...,57.64,0.0042,1.17,12.21,371.91,Newcastle | Australia,-32.9167,151.7500,2014,1
3,4.98,19.61,2.52,7.41,9.93,4.67,0.21,2.05,45.29,1.39,...,50.29,0.0025,1.16,4.09,5.00,Wollongong | Australia,-34.4331,150.8831,2014,1
4,4.94,9.70,4.26,12.23,16.49,0.46,0.26,2.91,46.39,1.12,...,45.25,0.0041,1.33,5.44,5.00,Central Coast | Australia,-33.2992,151.1922,2014,1


In [33]:
df.describe()

,PM2.5 (ug/m3),PM10 (ug/m3),NO (ug/m3),NO2 (ug/m3),NOx (ppb),NH3 (ug/m3),CO (mg/m3),SO2 (ug/m3),O3 (ug/m3),Benzene (ug/m3),...,Wind_Speed (km/h),Humidity (%),Deforestation_Rate_%,Industry_Growth_%,CO2_Emission_MT,Population_Density_per_SqKm,Lat,Lon,Year,Month
count,251280.000000,251280.000000,251280.000000,251280.000000,251280.000000,251280.000000,251280.000000,251280.000000,251280.000000,251280.000000,...,251280.000000,251280.000000,251280.00000,251280.000000,251280.000000,251280.000000,251280.000000,251280.000000,251280.000000,251280.000000
mean,34.837387,66.947389,13.876210,26.876397,27.827193,9.842750,0.978907,10.561013,44.447169,1.528019,...,11.279167,65.545506,0.51211,3.178577,39.717439,1051.710088,23.643245,50.470849,2019.500000,6.500000
std,45.411227,88.449117,13.744624,22.465462,24.161426,9.502556,1.174094,27.616103,21.694068,1.971817,...,5.133662,16.672722,0.69684,2.296709,71.076151,2566.740587,22.891177,76.671179,3.452059,3.452059
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,5.000000,0.00000,-4.200000,-3.060000,1.000000,-42.883300,-157.846000,2014.000000,1.000000
25%,12.520000,22.090000,5.890000,13.070000,13.290000,4.310000,0.370000,3.300000,30.240000,0.520000,...,7.580000,55.100000,0.02600,1.400000,3.834858,131.270000,11.674200,6.639400,2016.750000,3.750000
50%,20.360000,36.215167,9.790000,20.632091,21.130000,7.230000,0.610000,5.791850,40.850000,0.960000,...,10.700000,67.030000,0.23200,2.490000,9.850000,392.630000,29.025800,73.016700,2019.500000,6.500000
75%,36.270000,70.430000,16.630000,32.870000,33.730000,11.481103,1.080000,10.870000,54.219051,1.660000,...,14.333404,77.500000,0.69300,4.748989,35.520000,1105.366967,39.141300,115.083300,2022.250000,9.250000
max,800.000000,1547.700000,155.496505,381.890797,283.297624,91.550000,15.000000,1500.000000,200.000000,30.000000,...,48.300000,100.000000,8.08600,11.999881,697.060000,28044.860054,69.333300,153.400000,2025.000000,12.000000


In [34]:
df["Seq_id"] = df["Year"]*12 + df["Month"]

In [35]:
first_cols = ['City_id', 'Seq_id', 'Year', 'Month', 'Lat', 'Lon', 'Population_Density_per_SqKm']

df = df[first_cols + [c for c in df.columns if c not in first_cols]]

In [36]:
df.dtypes

City_id                         object
Seq_id                           int32
Year                             int32
Month                            int32
Lat                            float64
Lon                            float64
Population_Density_per_SqKm    float64
PM2.5 (ug/m3)                  float64
PM10 (ug/m3)                   float64
NO (ug/m3)                     float64
NO2 (ug/m3)                    float64
NOx (ppb)                      float64
NH3 (ug/m3)                    float64
CO (mg/m3)                     float64
SO2 (ug/m3)                    float64
O3 (ug/m3)                     float64
Benzene (ug/m3)                float64
Toluene (ug/m3)                float64
Xylene (ug/m3)                 float64
AQI                            float64
Wind_Speed (km/h)              float64
Humidity (%)                   float64
Deforestation_Rate_%           float64
Industry_Growth_%              float64
CO2_Emission_MT                float64
dtype: object